<a href="https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

**Chosen lane:** Refresh / Content Opportunity Scoring (one of FlyRank's four predefined lanes)  
**Purpose:** Choose a useful problem to study over the next seven weeks; use observed starter data to justify it, **not** to claim a new trained model.

[Open this notebook in Google Colab from my GitHub repository](https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Core first, AI second:** I will read, check, and personalize this framing before submission. The conclusions below describe a *proposed decision-support research project*, not proven SEO causation or a production system.

This research question is consistent with my [ML-03 framing notebook](https://github.com/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb).

## 1. My lane (or freestyle) and why

I provisionally choose **Refresh / Content Opportunity Scoring**. I want to investigate how search visibility, content freshness, and engagement signals can help decide **which existing pages should be reviewed first** when an editorial team cannot review everything. The output would be a **ranked queue for human review**, rather than automatic edits or publishing.

I chose this lane because it builds on the starter notebooks' comparison of hand-written rules and ML-assisted rankings, and it can be grounded in the provided, pseudonymized content-performance data. I can still revise the lane by the end of Week 4 if the data does not support it.

**Unit of analysis:** One pseudonymized content page, aggregated over the starter dataset's trailing 90-day window. The starter data is a snapshot; it cannot, by itself, establish whether a recommended edit would *cause* a future traffic improvement.

In [1]:
# Section 1 — Load the same safe starter dataset used in ML-01 and ML-03.
# In a fresh Colab runtime, clone MY public template copy (no credentials needed).
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

def locate_repo():
    if "google.colab" in sys.modules:
        root = Path("/content/flyrank-ml-internship")
        if not (root / DATA_REL).is_file():
            if root.exists():
                raise FileNotFoundError("Repository folder exists but the starter CSV is missing.")
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/Jhonas05/flyrank-ml-internship.git", str(root)],
                check=True,
            )
        return root
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / DATA_REL).is_file():
            return candidate
    raise FileNotFoundError("Open in Colab or run from a local clone of the FlyRank repo.")

repo_root = locate_repo()
df = pd.read_csv(repo_root / DATA_REL)
assert df["content_id"].is_unique, "Expected one record per content page."
print(f"Loaded starter CSV: {len(df):,} pages x {df.shape[1]} columns")
print("Unit of analysis: one pseudonymized content page per row")

Loaded starter CSV: 30,000 pages x 44 columns
Unit of analysis: one pseudonymized content page per row


## 2. The question: decision, action, cost of a wrong call

**Research question:** *How can search performance, content freshness, and engagement signals help prioritize existing content pages for editorial review more effectively than a transparent fixed-rule ranking?*

- **Decision improved:** Which pages should a content editor review **first** under limited time?
- **Who acts:** A content editor or SEO analyst.
- **Action:** Inspect high-ranked pages for outdated information, relevance to search intent, presentation, and opportunities for appropriate revisions. A high score **does not** automatically authorize an edit.
- **Cost of false positives:** An editor spends time reviewing a page that did not need attention.
- **Cost of false negatives:** A page that could benefit from timely review is overlooked.
- **Why data/ML may help:** Freshness, search visibility, click behavior, position, and engagement can interact. A simple rule remains the **baseline** to beat; ML is only justified if a fairly evaluated approach provides a meaningful improvement.

**Provisional measurement:** We will consider **Precision@50** for a top-50 review queue. The starter's `trend_direction == "down"` label is a **current-window, rule-defined proxy** for testing the workflow, *not a future outcome*. Later work should use time-aligned, subsequently observed outcomes and client-holdout validation before claiming predictive performance.

In [2]:
# Section 2 — Show the actual data grain without publishing any identifiers.
preview_fields = [
    "content_type", "impressions_90d", "clicks_90d",
    "days_since_last_update", "content_age_days", "trend_direction"
]
print("Example page-level observations (identifiers hidden):")
display(df[preview_fields].head(5).reset_index(drop=True))
print("Suggested human action: review prioritized pages; do not auto-edit or auto-publish.")

Example page-level observations (identifiers hidden):


,content_type,impressions_90d,clicks_90d,days_since_last_update,content_age_days,trend_direction
0,keyword article,3803,29,20,187,down
1,keyword article,15320,7,25,445,down
2,keyword article,12581,11,20,141,down
3,keyword article,11751,58,22,463,stable
4,keyword article,19140,24,14,263,down


Suggested human action: review prioritized pages; do not auto-edit or auto-publish.


## 3. Quick look at the data (2–3 real numbers)

I will compute three basic observations from **`data/raw/content_refresh_anonymized.csv`**, which is the provided 90-day starter snapshot:

1. **Total pages:** the scale of the starter dataset.
2. **Eligible pages:** pages with at least **100 impressions in 90 days** and at least **90 days of content age**, consistent with my provisional ML-03 review scope. The 100-impression cutoff is an *initial operational choice*, not a universal SEO law.
3. **Pages with a declining proxy:** among eligible pages, how many have `trend_direction == "down"`? This is an *observed current-window category* defined from an impressions trend rule; it is **not a verified future decline**.

These counts will tell me whether there is a sufficiently large group of pages for prioritization research. They do not tell me that ML has already improved outcomes.

In [3]:
# Section 3 — Recompute three supporting numbers directly from the starter CSV.
eligible = df.loc[
    (df["impressions_90d"] >= 100) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset=["content_id"]).copy()

eligible["declining_proxy"] = (
    eligible["trend_direction"].astype(str).str.lower().eq("down")
).astype("int8")

total_pages = len(df)
eligible_pages = len(eligible)
declining_pages = int(eligible["declining_proxy"].sum())

print(f"1. Total content pages:                    {total_pages:,}")
print(f"2. Pages eligible for editorial review:   {eligible_pages:,}")
print(f"3. Eligible pages with declining proxy:   {declining_pages:,}")
print(f"   Share of eligible pages:              {declining_pages / eligible_pages:.1%}")
assert eligible["content_id"].is_unique
assert 0 <= declining_pages <= eligible_pages <= total_pages

print("\nThese are descriptive starter-data counts, not model-validation results.")

1. Total content pages:                    30,000
2. Pages eligible for editorial review:   22,006
3. Eligible pages with declining proxy:   13,152
   Share of eligible pages:              59.8%

These are descriptive starter-data counts, not model-validation results.


## 4. Careful words: what I can and can't claim

**What I can say:** The anonymized starter snapshot contains measurable variation across content pages. I can report observed counts, directional associations, and the performance of a clearly defined ranking method against a fair baseline. My proposed product is a **decision-support queue** to help humans prioritize page reviews.

**What I cannot say:** This snapshot alone does **not** prove that a content update causes higher search rankings or traffic, or that a model predicts Google's algorithm. I cannot claim ML outperforms a fixed rule without appropriate evaluation, and I cannot report a current-window proxy result as future predictive accuracy.

**Specific precautions:**

- `trend_direction` comes from `trend_pct` and is the *proxy label*. Neither field may enter the predictive feature set for that proxy. Other trend-input fields must also be audited for label leakage.
- `content_id` and `client_id` are pseudonymous **grouping** keys, not predictive features or published identities; any future train/test split should hold out clients.
- Feature windows must be earlier than any future-outcome label window. A 90-day snapshot can overlap the supposed outcome period.
- `ctr` is stored in percentage units (e.g., `0.76` means 0.76%), while `avg_position == 0` means missing position data. Missing fields cannot automatically be interpreted as real zeros.
- All reports and shared notebook outputs must avoid private client names, domains, URLs, and search queries.

In [4]:
# Section 4 — Lightweight data-contract checks; no model is trained here.
label_source_fields = {"trend_direction", "trend_pct",
                       "impressions_last_30d", "impressions_prev_30d"}
ids = {"content_id", "client_id"}
# These are examples of fields for *future investigation*, NOT an approved
# leakage-safe feature set. They require time-window review before modeling.
feature_ideas = {"content_age_days", "days_since_last_update",
                 "impressions_90d", "ctr", "avg_position", "engagement_rate"}
assert not feature_ideas.intersection(label_source_fields | ids)

print("Research notebook only: no ML model trained or validated here.")
print("Starter data checks:")
print(f"- Unique page rows: {df['content_id'].is_unique}")
print(f"- Rows with avg_position == 0 (no position data): {(df['avg_position'] == 0).sum():,}")
print(f"- Rows with missing search_volume: {df['search_volume'].isna().sum():,}")
print("- Predictive feature windows and client-holdout design need review later.")

Research notebook only: no ML model trained or validated here.
Starter data checks:
- Unique page rows: True
- Rows with avg_position == 0 (no position data): 1,205
- Rows with missing search_volume: 2,468
- Predictive feature windows and client-holdout design need review later.


## Self-check

Before submitting, **personally verify** the items below (tick them in Colab after reviewing the content):

- [ ] I can explain *why* I chose Refresh / Content Opportunity Scoring, and that this lane is provisional.
- [ ] I can state the research question, the page-level unit, the decision, the human action, and the cost of being wrong.
- [ ] My Section 3 code ran successfully and produced at least two real numbers from the starter data.
- [ ] I understand that the starter `trend_direction` label is a **current-window proxy**, not a future outcome.
- [ ] I used observed / directional / decision-support wording, not causal or "predict Google" claims.
- [ ] The notebook ran top-to-bottom without errors and no private client details, tokens, or extra datasets entered GitHub.
- [ ] I read and personalized my framing, saved executed outputs at `work/notebooks/w01_research_question.ipynb`, and confirmed them on github.com.

**One-paragraph framing statement (review and personalize):** For content editors deciding which existing pages to review first, I propose studying a ranked review queue based on pseudonymized search, freshness, and engagement signals. I will compare a learned ranking with a transparent fixed-rule baseline using an explicitly defined Precision@50 metric, while accounting for client separation and time-window leakage in later evaluation. Wrong calls waste review time or miss potentially declining pages. The intended output supports editorial judgment rather than automatic edits or causal claims about search traffic.